# Detección y subtipificación de leucemia linfoblástica aguda usando IA

**Notebook final unificado ejecutado directamente desde Kaggle**

Este notebook descarga el dataset **Acute Lymphoblastic Leukemia (ALL) image dataset** desde Kaggle (`mehradaria/leukemia`) y ejecuta el pipeline completo sin cargar archivos CSV generados previamente.

Clases del problema:

- `Benign`
- `Early`
- `Pre`
- `Pro`

El objetivo académico es evaluar técnicas de inteligencia artificial sobre imágenes médicas de frotis de sangre periférica: exploración, extracción de características manuales, análisis estadístico, clasificación supervisada clásica, una red neuronal profunda densa y aprendizaje no supervisado.

Importante: este proyecto es una herramienta de análisis computacional y apoyo académico. El diagnóstico clínico definitivo de leucemia linfoblástica aguda requiere evaluación médica, pruebas de laboratorio y citometría de flujo.


## 1. Configuración general

Esta versión está diseñada para Google Colab o Jupyter. No carga `leukemia_handcrafted_features.csv`, `model_comparison_results.csv` ni otros CSVs previos. Si se guardan CSVs al final, corresponden a la ejecución actual.


In [ ]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Instalando {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


required_packages = {
    "kagglehub": "kagglehub",
    "cv2": "opencv-python-headless",
    "skimage": "scikit-image",
    "tqdm": "tqdm",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}

for import_name, pip_name in required_packages.items():
    ensure_package(import_name, pip_name)


In [ ]:
import os
import random
import warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

import cv2
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from IPython.display import display, Markdown

from skimage.measure import label, regionprops
from skimage.measure import shannon_entropy
from skimage.feature import local_binary_pattern, hog

try:
    from skimage.feature import graycomatrix, graycoprops
except ImportError:
    from skimage.feature import greycomatrix as graycomatrix
    from skimage.feature import greycoprops as graycoprops

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CLASS_ORDER = ["Benign", "Early", "Pre", "Pro"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs" / "kaggle_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_SUBSET = False
MAX_IMAGES_PER_CLASS = 250
IMAGE_SIZE = (224, 224)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13


## 2. Descarga del dataset desde Kaggle

Se usa `kagglehub.dataset_download("mehradaria/leukemia")`. En Colab normalmente descarga el dataset público sin configurar rutas manuales. Si Kaggle solicita credenciales en otro entorno, se debe configurar la API de Kaggle.


In [ ]:
DATASET_SLUG = "mehradaria/leukemia"
dataset_path = Path(kagglehub.dataset_download(DATASET_SLUG))

print("Dataset descargado/localizado en:")
print(dataset_path)

print("\nPrimeros elementos encontrados:")
for path in sorted(dataset_path.iterdir())[:20]:
    print("-", path.relative_to(dataset_path))


## 3. Descubrimiento de imágenes originales y segmentadas

El dataset contiene imágenes originales y segmentadas. El siguiente bloque recorre la estructura de carpetas, infiere la clase desde la ruta o el nombre del archivo y empareja cada imagen original con su segmentación cuando está disponible.


In [ ]:
def infer_class_from_path(path):
    parts_lower = [part.lower() for part in path.parts]
    name_lower = path.name.lower()
    for cls in CLASS_ORDER:
        cls_lower = cls.lower()
        if cls_lower in parts_lower or cls_lower in name_lower:
            return cls
    return None


def is_segmented_path(path):
    tokens = [part.lower() for part in path.parts]
    joined = " ".join(tokens)
    return "segmented" in joined or "segment" in joined


def is_original_path(path):
    tokens = [part.lower() for part in path.parts]
    joined = " ".join(tokens)
    return "original" in joined or "orig" in joined


all_image_files = [
    path for path in dataset_path.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]

original_files = []
segmented_files = []

for path in all_image_files:
    cls = infer_class_from_path(path)
    if cls is None:
        continue
    if is_segmented_path(path):
        segmented_files.append(path)
    elif is_original_path(path):
        original_files.append(path)

if not original_files:
    original_files = [
        path for path in all_image_files
        if infer_class_from_path(path) is not None and not is_segmented_path(path)
    ]

if not original_files and segmented_files:
    print("Aviso: no se encontraron originales; se usarán segmentadas como entrada visual.")
    original_files = segmented_files.copy()

segmented_by_key = {}
for path in segmented_files:
    cls = infer_class_from_path(path)
    segmented_by_key[(cls, path.stem.lower())] = path

rows = []
for original_path in sorted(original_files):
    cls = infer_class_from_path(original_path)
    if cls is None:
        continue
    key = (cls, original_path.stem.lower())
    segmented_path = segmented_by_key.get(key)
    img = cv2.imread(str(original_path))
    height, width = (None, None)
    if img is not None:
        height, width = img.shape[:2]
    rows.append({
        "class": cls,
        "file_name": original_path.name,
        "stem": original_path.stem,
        "original_path": str(original_path),
        "segmented_path": str(segmented_path) if segmented_path is not None else None,
        "has_segmented": segmented_path is not None,
        "width": width,
        "height": height,
    })

df_images = pd.DataFrame(rows)
df_images["class"] = pd.Categorical(df_images["class"], CLASS_ORDER, ordered=True)
df_images = df_images.sort_values(["class", "file_name"]).reset_index(drop=True)

if len(df_images) == 0:
    raise RuntimeError(
        "No se detectaron imágenes válidas del dataset. Revisa la descarga de Kaggle "
        "y la estructura de carpetas Original/Segmented."
    )

print("Imágenes totales detectadas:", len(df_images))
print("Originales detectadas:", len(original_files))
print("Segmentadas detectadas:", len(segmented_files))
display(df_images.head())


In [ ]:
class_summary = (
    df_images["class"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="n_images")
)
class_summary["class"] = pd.Categorical(class_summary["class"], CLASS_ORDER, ordered=True)
class_summary = class_summary.sort_values("class").reset_index(drop=True)
class_summary["proportion"] = class_summary["n_images"] / class_summary["n_images"].sum()

display(class_summary.assign(proportion=lambda d: d["proportion"].round(3)))

plt.figure(figsize=(7, 4))
ax = sns.barplot(data=class_summary, x="class", y="n_images", palette="Set2")
for i, row in class_summary.iterrows():
    ax.text(i, row["n_images"] + max(5, class_summary["n_images"].max() * 0.01), str(row["n_images"]), ha="center")
plt.title("Distribución de imágenes por clase")
plt.xlabel("Clase")
plt.ylabel("Número de imágenes")
plt.tight_layout()
plt.show()


In [ ]:
def read_rgb(path):
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        return None
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(2, len(CLASS_ORDER), figsize=(4 * len(CLASS_ORDER), 7))
for j, cls in enumerate(CLASS_ORDER):
    subset = df_images[df_images["class"].astype(str) == cls]
    if len(subset) == 0:
        axes[0, j].axis("off")
        axes[1, j].axis("off")
        continue
    row = subset.iloc[0]
    original = read_rgb(row["original_path"])
    segmented = read_rgb(row["segmented_path"]) if isinstance(row["segmented_path"], str) else None

    axes[0, j].imshow(original)
    axes[0, j].set_title(f"{cls} - original")
    axes[0, j].axis("off")

    if segmented is not None:
        axes[1, j].imshow(segmented)
        axes[1, j].set_title(f"{cls} - segmentada")
    else:
        axes[1, j].text(0.5, 0.5, "Sin segmentación", ha="center", va="center")
    axes[1, j].axis("off")

plt.tight_layout()
plt.show()


## 4. Extracción de características handcrafted

Se extraen variables manuales de:

- Color en RGB, HSV y LAB.
- Intensidad y escala de grises.
- Textura GLCM y LBP.
- Bordes con Canny.
- Gradientes HOG.
- Frecuencia DCT.
- Morfología desde la máscara segmentada.

La extracción se hace desde cero directamente sobre las imágenes descargadas de Kaggle.


In [ ]:
def safe_float(value):
    try:
        if value is None or np.isnan(value) or np.isinf(value):
            return 0.0
        return float(value)
    except Exception:
        return 0.0


def resize_image(img, size=IMAGE_SIZE):
    return cv2.resize(img, size, interpolation=cv2.INTER_AREA)


def create_mask_from_segmented(seg_img):
    if seg_img is None:
        return None
    if len(seg_img.shape) == 3:
        gray_seg = cv2.cvtColor(seg_img, cv2.COLOR_BGR2GRAY)
    else:
        gray_seg = seg_img.copy()
    _, mask = cv2.threshold(gray_seg, 10, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask


def get_roi_pixels(channel, mask):
    if mask is None:
        return channel.flatten()
    pixels = channel[mask > 0]
    if len(pixels) == 0:
        return channel.flatten()
    return pixels


def extract_color_features(img_rgb, mask=None, prefix=""):
    features = {}
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    color_spaces = {
        "rgb": (img_rgb, ["r", "g", "b"]),
        "hsv": (hsv, ["h", "s", "v"]),
        "lab": (lab, ["l", "a", "b"]),
    }
    for space_name, (space_img, channels) in color_spaces.items():
        for i, channel_name in enumerate(channels):
            pixels = get_roi_pixels(space_img[:, :, i], mask)
            features[f"{prefix}{space_name}_{channel_name}_mean"] = safe_float(np.mean(pixels))
            features[f"{prefix}{space_name}_{channel_name}_std"] = safe_float(np.std(pixels))
            features[f"{prefix}{space_name}_{channel_name}_min"] = safe_float(np.min(pixels))
            features[f"{prefix}{space_name}_{channel_name}_max"] = safe_float(np.max(pixels))
            features[f"{prefix}{space_name}_{channel_name}_median"] = safe_float(np.median(pixels))
            features[f"{prefix}{space_name}_{channel_name}_q25"] = safe_float(np.percentile(pixels, 25))
            features[f"{prefix}{space_name}_{channel_name}_q75"] = safe_float(np.percentile(pixels, 75))
    for space_name, space_img in {"rgb": img_rgb, "hsv": hsv, "lab": lab}.items():
        for i in range(3):
            pixels = get_roi_pixels(space_img[:, :, i], mask)
            hist, _ = np.histogram(pixels, bins=16, range=(0, 256), density=True)
            for j, val in enumerate(hist):
                features[f"{prefix}{space_name}_ch{i}_hist_{j}"] = safe_float(val)
    return features


def extract_gray_features(gray, mask=None, prefix=""):
    features = {}
    pixels = get_roi_pixels(gray, mask)
    features[f"{prefix}gray_mean"] = safe_float(np.mean(pixels))
    features[f"{prefix}gray_std"] = safe_float(np.std(pixels))
    features[f"{prefix}gray_min"] = safe_float(np.min(pixels))
    features[f"{prefix}gray_max"] = safe_float(np.max(pixels))
    features[f"{prefix}gray_median"] = safe_float(np.median(pixels))
    features[f"{prefix}gray_q25"] = safe_float(np.percentile(pixels, 25))
    features[f"{prefix}gray_q75"] = safe_float(np.percentile(pixels, 75))
    features[f"{prefix}gray_iqr"] = safe_float(np.percentile(pixels, 75) - np.percentile(pixels, 25))
    features[f"{prefix}gray_entropy"] = safe_float(shannon_entropy(pixels))
    features[f"{prefix}brightness"] = safe_float(np.mean(pixels))
    features[f"{prefix}contrast"] = safe_float(np.std(pixels))
    hist, _ = np.histogram(pixels, bins=32, range=(0, 256), density=True)
    for i, val in enumerate(hist):
        features[f"{prefix}gray_hist_{i}"] = safe_float(val)
    return features


def extract_glcm_features(gray, mask=None, prefix=""):
    features = {}
    gray_small = cv2.resize(gray, (96, 96), interpolation=cv2.INTER_AREA)
    if mask is not None:
        mask_small = cv2.resize(mask, (96, 96), interpolation=cv2.INTER_NEAREST)
        gray_small = gray_small.copy()
        gray_small[mask_small == 0] = 0
    quant = (gray_small // 16).astype(np.uint8)
    glcm = graycomatrix(
        quant,
        distances=[1, 2, 4],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=16,
        symmetric=True,
        normed=True,
    )
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        vals = graycoprops(glcm, prop).flatten()
        features[f"{prefix}glcm_{prop}_mean"] = safe_float(np.mean(vals))
        features[f"{prefix}glcm_{prop}_std"] = safe_float(np.std(vals))
        features[f"{prefix}glcm_{prop}_min"] = safe_float(np.min(vals))
        features[f"{prefix}glcm_{prop}_max"] = safe_float(np.max(vals))
    return features


def extract_lbp_features(gray, mask=None, prefix=""):
    features = {}
    gray_small = cv2.resize(gray, (128, 128), interpolation=cv2.INTER_AREA)
    radius = 2
    n_points = 8 * radius
    lbp = local_binary_pattern(gray_small, P=n_points, R=radius, method="uniform")
    if mask is not None:
        mask_small = cv2.resize(mask, (128, 128), interpolation=cv2.INTER_NEAREST)
        lbp_pixels = lbp[mask_small > 0]
        if len(lbp_pixels) == 0:
            lbp_pixels = lbp.flatten()
    else:
        lbp_pixels = lbp.flatten()
    hist, _ = np.histogram(lbp_pixels, bins=n_points + 2, range=(0, n_points + 2), density=True)
    for i, val in enumerate(hist):
        features[f"{prefix}lbp_hist_{i}"] = safe_float(val)
    features[f"{prefix}lbp_mean"] = safe_float(np.mean(lbp_pixels))
    features[f"{prefix}lbp_std"] = safe_float(np.std(lbp_pixels))
    features[f"{prefix}lbp_entropy"] = safe_float(shannon_entropy(lbp_pixels))
    return features


def extract_hog_features(gray, prefix=""):
    features = {}
    gray_small = cv2.resize(gray, (128, 128), interpolation=cv2.INTER_AREA)
    hog_vec = hog(
        gray_small,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    )
    features[f"{prefix}hog_mean"] = safe_float(np.mean(hog_vec))
    features[f"{prefix}hog_std"] = safe_float(np.std(hog_vec))
    features[f"{prefix}hog_min"] = safe_float(np.min(hog_vec))
    features[f"{prefix}hog_max"] = safe_float(np.max(hog_vec))
    features[f"{prefix}hog_energy"] = safe_float(np.sum(hog_vec ** 2))
    features[f"{prefix}hog_entropy"] = safe_float(shannon_entropy(hog_vec))
    return features


def extract_edge_features(gray, mask=None, prefix=""):
    features = {}
    edges = cv2.Canny(gray, 100, 200)
    if mask is not None:
        roi_edges = edges[mask > 0]
        roi_area = np.sum(mask > 0)
    else:
        roi_edges = edges.flatten()
        roi_area = gray.shape[0] * gray.shape[1]
    if roi_area == 0:
        roi_area = gray.shape[0] * gray.shape[1]
    features[f"{prefix}edge_density"] = safe_float(np.sum(roi_edges > 0) / roi_area)
    features[f"{prefix}edge_mean_intensity"] = safe_float(np.mean(roi_edges))
    return features


def extract_dct_features(gray, mask=None, prefix=""):
    features = {}
    gray_small = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32)
    if mask is not None:
        mask_small = cv2.resize(mask, (64, 64), interpolation=cv2.INTER_NEAREST)
        gray_small[mask_small == 0] = 0
    dct_img = cv2.dct(gray_small)
    low_freq = dct_img[:8, :8].flatten()
    features[f"{prefix}dct_low_mean"] = safe_float(np.mean(low_freq))
    features[f"{prefix}dct_low_std"] = safe_float(np.std(low_freq))
    features[f"{prefix}dct_low_energy"] = safe_float(np.sum(low_freq ** 2))
    for i in range(8):
        for j in range(8):
            features[f"{prefix}dct_{i}_{j}"] = safe_float(dct_img[i, j])
    return features


def extract_morphological_features(mask, prefix=""):
    features = {}
    morph_names = [
        "mask_area_ratio", "mask_area", "mask_perimeter",
        "mask_circularity", "mask_eccentricity", "mask_solidity",
        "mask_extent", "mask_major_axis_length", "mask_minor_axis_length",
        "mask_bbox_area", "mask_equivalent_diameter",
    ]
    if mask is None:
        for name in morph_names:
            features[f"{prefix}{name}"] = 0.0
        for i in range(7):
            features[f"{prefix}hu_moment_{i}"] = 0.0
        return features
    binary = (mask > 0).astype(np.uint8)
    total_area = binary.shape[0] * binary.shape[1]
    area = np.sum(binary > 0)
    features[f"{prefix}mask_area_ratio"] = safe_float(area / total_area)
    features[f"{prefix}mask_area"] = safe_float(area)
    labeled = label(binary)
    props = regionprops(labeled)
    if len(props) == 0:
        for name in morph_names[2:]:
            features[f"{prefix}{name}"] = 0.0
    else:
        region = max(props, key=lambda r: r.area)
        perimeter = region.perimeter
        region_area = region.area
        circularity = 4 * np.pi * region_area / (perimeter ** 2) if perimeter > 0 else 0.0
        minr, minc, maxr, maxc = region.bbox
        bbox_area = (maxr - minr) * (maxc - minc)
        features[f"{prefix}mask_perimeter"] = safe_float(perimeter)
        features[f"{prefix}mask_circularity"] = safe_float(circularity)
        features[f"{prefix}mask_eccentricity"] = safe_float(region.eccentricity)
        features[f"{prefix}mask_solidity"] = safe_float(region.solidity)
        features[f"{prefix}mask_extent"] = safe_float(region.extent)
        features[f"{prefix}mask_major_axis_length"] = safe_float(region.major_axis_length)
        features[f"{prefix}mask_minor_axis_length"] = safe_float(region.minor_axis_length)
        features[f"{prefix}mask_bbox_area"] = safe_float(bbox_area)
        features[f"{prefix}mask_equivalent_diameter"] = safe_float(region.equivalent_diameter)
    moments = cv2.moments(binary)
    hu = cv2.HuMoments(moments).flatten()
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    for i, val in enumerate(hu_log):
        features[f"{prefix}hu_moment_{i}"] = safe_float(val)
    return features


def extract_all_features(original_path, segmented_path=None, image_size=IMAGE_SIZE):
    features = {}
    img_bgr = cv2.imread(str(original_path))
    if img_bgr is None:
        raise ValueError(f"No se pudo leer la imagen: {original_path}")
    img_bgr = resize_image(img_bgr, image_size)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    mask = None
    if segmented_path is not None and isinstance(segmented_path, str) and Path(segmented_path).exists():
        seg_bgr = cv2.imread(str(segmented_path))
        if seg_bgr is not None:
            seg_bgr = resize_image(seg_bgr, image_size)
            mask = create_mask_from_segmented(seg_bgr)

    features.update(extract_color_features(img_rgb, mask=None, prefix="global_"))
    features.update(extract_gray_features(gray, mask=None, prefix="global_"))
    features.update(extract_glcm_features(gray, mask=None, prefix="global_"))
    features.update(extract_lbp_features(gray, mask=None, prefix="global_"))
    features.update(extract_hog_features(gray, prefix="global_"))
    features.update(extract_edge_features(gray, mask=None, prefix="global_"))
    features.update(extract_dct_features(gray, mask=None, prefix="global_"))

    features.update(extract_color_features(img_rgb, mask=mask, prefix="roi_"))
    features.update(extract_gray_features(gray, mask=mask, prefix="roi_"))
    features.update(extract_glcm_features(gray, mask=mask, prefix="roi_"))
    features.update(extract_lbp_features(gray, mask=mask, prefix="roi_"))
    features.update(extract_edge_features(gray, mask=mask, prefix="roi_"))
    features.update(extract_dct_features(gray, mask=mask, prefix="roi_"))
    features.update(extract_morphological_features(mask, prefix="seg_"))
    return features


In [ ]:
if USE_SUBSET:
    parts = []
    for cls in CLASS_ORDER:
        subset = df_images[df_images["class"].astype(str) == cls]
        parts.append(subset.sample(n=min(MAX_IMAGES_PER_CLASS, len(subset)), random_state=SEED))
    df_work = pd.concat(parts).reset_index(drop=True)
else:
    df_work = df_images.copy().reset_index(drop=True)

print("Imágenes a procesar:", len(df_work))
print("USE_SUBSET =", USE_SUBSET)

feature_rows = []
errors = []

for _, row in tqdm(df_work.iterrows(), total=len(df_work), desc="Extrayendo features"):
    try:
        feats = extract_all_features(
            original_path=row["original_path"],
            segmented_path=row["segmented_path"],
            image_size=IMAGE_SIZE,
        )
        for col in ["class", "file_name", "stem", "original_path", "segmented_path", "has_segmented", "width", "height"]:
            feats[col] = row[col]
        feature_rows.append(feats)
    except Exception as exc:
        errors.append((row["original_path"], repr(exc)))

df_features = pd.DataFrame(feature_rows)
print("Shape final de features:", df_features.shape)
print("Errores de procesamiento:", len(errors))
if errors:
    display(pd.DataFrame(errors, columns=["path", "error"]).head(10))

display(df_features.head())

features_output = OUTPUT_DIR / "leukemia_handcrafted_features_generated.csv"
df_features.to_csv(features_output, index=False)
print("CSV de features generado en esta ejecución:", features_output)


## 5. Limpieza de features y resumen por familias

Se eliminan variables constantes y se imputan valores faltantes. Esto evita que columnas sin variación afecten ranking, escalamiento o entrenamiento.


In [ ]:
metadata_cols = ["class", "file_name", "stem", "original_path", "segmented_path", "has_segmented"]

numeric_feature_cols = [
    col for col in df_features.columns
    if col not in metadata_cols and pd.api.types.is_numeric_dtype(df_features[col])
]

X_raw = df_features[numeric_feature_cols].copy()
y = df_features["class"].astype(str).copy()

X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
nan_before = int(X_raw.isna().sum().sum())
X_filled = X_raw.fillna(X_raw.median(numeric_only=True)).fillna(0)
constant_cols = [col for col in X_filled.columns if X_filled[col].nunique(dropna=False) <= 1]
X_clean = X_filled.drop(columns=constant_cols)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)

prep_summary = pd.DataFrame({
    "Concepto": [
        "Muestras procesadas",
        "Variables numéricas candidatas",
        "NaN antes de imputar",
        "Variables constantes eliminadas",
        "Variables finales",
        "Clases codificadas",
    ],
    "Valor": [
        len(df_features),
        len(numeric_feature_cols),
        nan_before,
        len(constant_cols),
        X_clean.shape[1],
        ", ".join(f"{i}: {name}" for i, name in enumerate(class_names)),
    ],
})
display(prep_summary)


In [ ]:
def categorize_feature(feature_name):
    if feature_name.startswith("global_"):
        scope = "Global"
    elif feature_name.startswith("roi_"):
        scope = "ROI segmentada"
    elif feature_name.startswith("seg_"):
        scope = "Morfología"
    else:
        scope = "Otros"
    name = feature_name.lower()
    if "rgb" in name:
        group = "Color RGB"
    elif "hsv" in name:
        group = "Color HSV"
    elif "lab" in name:
        group = "Color LAB"
    elif "gray" in name or "brightness" in name or "contrast" in name:
        group = "Intensidad / gris"
    elif "glcm" in name:
        group = "Textura GLCM"
    elif "lbp" in name:
        group = "Textura LBP"
    elif "hog" in name:
        group = "Gradientes HOG"
    elif "dct" in name:
        group = "Frecuencia DCT"
    elif "edge" in name:
        group = "Bordes"
    elif name.startswith("seg_") or "mask_" in name or "hu_moment" in name:
        group = "Morfología"
    else:
        group = "Otros"
    return scope, group


feature_families = pd.DataFrame(
    [(*categorize_feature(col), col) for col in X_clean.columns],
    columns=["scope", "feature_group", "feature"],
)
family_summary = (
    feature_families
    .groupby(["scope", "feature_group"])
    .size()
    .reset_index(name="n_features")
    .sort_values(["scope", "feature_group"])
)
display(family_summary)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=family_summary.sort_values("n_features", ascending=False),
    y="feature_group",
    x="n_features",
    hue="scope",
)
plt.title("Número de variables por familia")
plt.xlabel("Número de variables")
plt.ylabel("Familia")
plt.tight_layout()
plt.show()


## 6. Ranking estadístico de variables

Se combinan ANOVA F-score, información mutua e importancia de Random Forest. El ranking ayuda a interpretar qué familias de variables son más discriminativas y permite construir subconjuntos Top-N.


In [ ]:
scaler_rank = StandardScaler()
X_scaled_rank = scaler_rank.fit_transform(X_clean)

f_vals, p_vals = f_classif(X_scaled_rank, y_encoded)
mi_vals = mutual_info_classif(X_scaled_rank, y_encoded, random_state=SEED)

rf_selector = RandomForestClassifier(
    n_estimators=300,
    random_state=SEED,
    class_weight="balanced",
    n_jobs=-1,
)
rf_selector.fit(X_clean, y_encoded)

feature_ranking = pd.DataFrame({
    "feature": X_clean.columns,
    "anova_f": f_vals,
    "anova_pvalue": p_vals,
    "mutual_info": mi_vals,
    "rf_importance": rf_selector.feature_importances_,
})

feature_ranking["mean_rank_score"] = (
    feature_ranking["anova_f"].rank(ascending=False)
    + feature_ranking["mutual_info"].rank(ascending=False)
    + feature_ranking["rf_importance"].rank(ascending=False)
) / 3

feature_ranking = feature_ranking.sort_values("mean_rank_score").reset_index(drop=True)

ranking_output = OUTPUT_DIR / "leukemia_feature_ranking_generated.csv"
feature_ranking.to_csv(ranking_output, index=False)
print("Ranking generado en esta ejecución:", ranking_output)

display(feature_ranking.head(25).round(5))

top100_families = pd.DataFrame(
    [(*categorize_feature(feature), feature) for feature in feature_ranking.head(100)["feature"]],
    columns=["scope", "feature_group", "feature"],
)
top100_summary = (
    top100_families
    .groupby(["scope", "feature_group"])
    .size()
    .reset_index(name="n_top100")
    .sort_values("n_top100", ascending=False)
)
display(top100_summary)


**Interpretación.** Si el Top-N contiene variables de color, textura, morfología, frecuencia e intensidad, entonces las imágenes presentan diferencias cuantificables entre clases. Esta evidencia estadística debe validarse con modelos supervisados.


## 7. Clasificación supervisada clásica

Se entrena una batería de modelos con separación train/test estratificada. Se reportan métricas macro porque el dataset no está perfectamente balanceado y cada clase debe pesar de forma equivalente.


In [ ]:
def specificity_macro_score(y_true, y_pred, labels=None):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    specificities = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    return float(np.mean(specificities))


def evaluate_predictions(y_true, y_pred, model_name, feature_set, n_features):
    return {
        "feature_set": feature_set,
        "n_features": n_features,
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "specificity_macro": specificity_macro_score(y_true, y_pred, labels=np.arange(len(class_names))),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def build_models():
    return {
        "Gaussian Naive Bayes": Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB(var_smoothing=5.061576888752309e-07)),
        ]),
        "Decision Tree": Pipeline([
            ("model", DecisionTreeClassifier(
                criterion="entropy",
                max_depth=15,
                min_samples_leaf=4,
                min_samples_split=9,
                class_weight="balanced",
                random_state=SEED,
            )),
        ]),
        "Random Forest": Pipeline([
            ("model", RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
            )),
        ]),
        "SVM Linear": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="linear", C=1, class_weight="balanced", random_state=SEED)),
        ]),
        "SVM RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", random_state=SEED)),
        ]),
        "SVM Polynomial": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="poly", degree=3, C=1, gamma="scale", class_weight="balanced", random_state=SEED)),
        ]),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5)),
        ]),
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)),
        ]),
    }


X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_encoded,
    test_size=0.20,
    random_state=SEED,
    stratify=y_encoded,
)

feature_sets = {"all_features": list(X_clean.columns)}
for top_n in [30, 50, 100]:
    top_features = feature_ranking["feature"].head(min(top_n, X_clean.shape[1])).tolist()
    feature_sets[f"top{top_n}_features"] = top_features

model_results = []
trained_models = {}

for feature_set_name, feature_cols in feature_sets.items():
    print(f"\nFeature set: {feature_set_name} ({len(feature_cols)} variables)")
    for model_name, model in build_models().items():
        print("  Entrenando:", model_name)
        model.fit(X_train[feature_cols], y_train)
        y_pred = model.predict(X_test[feature_cols])
        model_results.append(evaluate_predictions(
            y_test,
            y_pred,
            model_name=model_name,
            feature_set=feature_set_name,
            n_features=len(feature_cols),
        ))
        if feature_set_name == "all_features":
            trained_models[model_name] = model

model_results_df = pd.DataFrame(model_results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(model_results_df.round(5))

model_results_output = OUTPUT_DIR / "model_comparison_results_generated.csv"
model_results_df.to_csv(model_results_output, index=False)
print("Resultados supervisados generados en:", model_results_output)


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=model_results_df, x="model", y="f1_macro", hue="feature_set")
plt.title("Comparación supervisada - F1 macro")
plt.xlabel("Modelo")
plt.ylabel("F1 macro")
plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

best_supervised = model_results_df.iloc[0]
best_model_name = best_supervised["model"]
best_feature_set = best_supervised["feature_set"]
best_feature_cols = feature_sets[best_feature_set]

best_model = build_models()[best_model_name]
best_model.fit(X_train[best_feature_cols], y_train)
y_pred_best = best_model.predict(X_test[best_feature_cols])

display(Markdown(f"**Mejor modelo:** {best_model_name} con `{best_feature_set}`."))
print(classification_report(y_test, y_pred_best, target_names=class_names, digits=4))

cm = confusion_matrix(y_test, y_pred_best, labels=np.arange(len(class_names)))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Matriz de confusión - {best_model_name}")
plt.xlabel("Predicción")
plt.ylabel("Clase real")
plt.tight_layout()
plt.show()


### Validación cruzada 10-fold

Se aplica sobre las Top 100 variables para reducir tiempo de cómputo y evaluar estabilidad.


In [ ]:
RUN_KFOLD = True

if RUN_KFOLD:
    top100_features = feature_sets["top100_features"]
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    scoring = {
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
    }
    kfold_rows = []
    for model_name, model in build_models().items():
        print("K-Fold:", model_name)
        scores = cross_validate(
            model,
            X_clean[top100_features],
            y_encoded,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
        )
        kfold_rows.append({
            "model": model_name,
            "feature_set": "top100_features",
            "accuracy_mean": scores["test_accuracy"].mean(),
            "accuracy_std": scores["test_accuracy"].std(),
            "precision_macro_mean": scores["test_precision_macro"].mean(),
            "recall_macro_mean": scores["test_recall_macro"].mean(),
            "f1_macro_mean": scores["test_f1_macro"].mean(),
            "f1_macro_std": scores["test_f1_macro"].std(),
        })
    kfold_results_df = pd.DataFrame(kfold_rows).sort_values("f1_macro_mean", ascending=False)
    display(kfold_results_df.round(5))
    kfold_output = OUTPUT_DIR / "kfold10_results_top100_generated.csv"
    kfold_results_df.to_csv(kfold_output, index=False)
    print("K-Fold generado en:", kfold_output)
else:
    print("RUN_KFOLD = False")


## 8. Red neuronal profunda densa

Se entrena una DNN densa con Keras usando las variables handcrafted, no imágenes crudas.

No se usa CNN, transfer learning ni autoencoders.


In [ ]:
RUN_DNN = True

tensorflow_available = True

if RUN_DNN:
    try:
        ensure_package("tensorflow", "tensorflow")
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras import layers
        from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    except Exception as exc:
        tensorflow_available = False
        RUN_DNN = False
        dnn_metrics = pd.DataFrame()
        print("No fue posible importar o instalar TensorFlow en este entorno.")
        print("La sección DNN se omite; el resto del notebook sigue siendo ejecutable.")
        print("Detalle:", repr(exc))

if RUN_DNN and tensorflow_available:
    tf.random.set_seed(SEED)

    scaler_dnn = StandardScaler()
    X_train_scaled = scaler_dnn.fit_transform(X_train)
    X_test_scaled = scaler_dnn.transform(X_test)

    model_dnn = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],), name="input_features"),
        layers.Dense(256, activation="relu", name="dense_256"),
        layers.Dropout(0.30, name="dropout_1"),
        layers.Dense(128, activation="relu", name="dense_128"),
        layers.Dropout(0.30, name="dropout_2"),
        layers.Dense(64, activation="relu", name="dense_64"),
        layers.Dense(len(class_names), activation="softmax", name="output_softmax"),
    ])
    model_dnn.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    model_dnn.summary()

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ]

    history = model_dnn.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=150,
        batch_size=32,
        callbacks=callbacks,
        verbose=1,
    )

    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(hist["loss"], label="train")
    axes[0].plot(hist["val_loss"], label="validación")
    axes[0].set_title("DNN - pérdida")
    axes[0].set_xlabel("Época")
    axes[0].legend()
    axes[1].plot(hist["accuracy"], label="train")
    axes[1].plot(hist["val_accuracy"], label="validación")
    axes[1].set_title("DNN - accuracy")
    axes[1].set_xlabel("Época")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    y_pred_dnn = np.argmax(model_dnn.predict(X_test_scaled), axis=1)
    dnn_metrics = pd.DataFrame([{
        "Model": "DNN features",
        "Input type": "Handcrafted features",
        "Number of input variables": X_train_scaled.shape[1],
        "Accuracy": accuracy_score(y_test, y_pred_dnn),
        "Precision macro": precision_score(y_test, y_pred_dnn, average="macro", zero_division=0),
        "Recall macro": recall_score(y_test, y_pred_dnn, average="macro", zero_division=0),
        "Specificity macro": specificity_macro_score(y_test, y_pred_dnn, labels=np.arange(len(class_names))),
        "F1 macro": f1_score(y_test, y_pred_dnn, average="macro", zero_division=0),
    }])
    display(dnn_metrics.round(5))
    print(classification_report(y_test, y_pred_dnn, target_names=class_names, digits=4))

    cm_dnn = confusion_matrix(y_test, y_pred_dnn, labels=np.arange(len(class_names)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_dnn, annot=True, fmt="d", cmap="Purples", xticklabels=class_names, yticklabels=class_names)
    plt.title("Matriz de confusión - DNN features")
    plt.xlabel("Predicción")
    plt.ylabel("Clase real")
    plt.tight_layout()
    plt.show()

    dnn_output = OUTPUT_DIR / "dnn_features_results_generated.csv"
    dnn_metrics.to_csv(dnn_output, index=False)
    print("Resultados DNN generados en:", dnn_output)
else:
    dnn_metrics = pd.DataFrame()
    print("RUN_DNN = False")


## 9. Aprendizaje no supervisado

Las etiquetas no se usan para entrenar PCA, K-Means, DBSCAN ni clustering jerárquico. Después de obtener clusters, se comparan con las clases reales mediante ARI, NMI, homogeneidad y completitud.


In [ ]:
scaler_unsup = StandardScaler()
X_scaled = scaler_unsup.fit_transform(X_clean)

pca_full = PCA(random_state=SEED)
pca_full.fit(X_scaled)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

n_80 = int(np.argmax(cumulative_var >= 0.80) + 1)
n_90 = int(np.argmax(cumulative_var >= 0.90) + 1)
n_95 = int(np.argmax(cumulative_var >= 0.95) + 1)

display(pd.DataFrame({
    "Varianza objetivo": ["80%", "90%", "95%"],
    "Componentes necesarios": [n_80, n_90, n_95],
}))

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cumulative_var) + 1), cumulative_var)
plt.axhline(0.95, color="red", linestyle="--", label="95%")
plt.title("PCA - varianza explicada acumulada")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada")
plt.legend()
plt.tight_layout()
plt.show()

pca_2d = PCA(n_components=2, random_state=SEED)
X_pca2 = pca_2d.fit_transform(X_scaled)
pca_3d = PCA(n_components=3, random_state=SEED)
X_pca3 = pca_3d.fit_transform(X_scaled)

print(f"PCA 2D: varianza explicada = {pca_2d.explained_variance_ratio_.sum():.2%}")
print(f"PCA 3D: varianza explicada = {pca_3d.explained_variance_ratio_.sum():.2%}")

pca_plot_df = pd.DataFrame({"PC1": X_pca2[:, 0], "PC2": X_pca2[:, 1], "class": y.values})
plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_plot_df, x="PC1", y="PC2", hue="class", hue_order=CLASS_ORDER, s=25, alpha=0.75)
plt.title("PCA 2D por clase real")
plt.tight_layout()
plt.show()


In [ ]:
pca_95 = PCA(n_components=n_95, random_state=SEED)
X_pca95 = pca_95.fit_transform(X_scaled)


def evaluate_clustering(X_space, y_true, labels_pred, model_name, space_name):
    mask_valid = labels_pred != -1
    n_clusters = len(set(labels_pred)) - (1 if -1 in labels_pred else 0)
    n_noise = int(np.sum(labels_pred == -1))

    if n_clusters >= 2 and np.sum(mask_valid) > n_clusters:
        sil = silhouette_score(X_space[mask_valid], labels_pred[mask_valid])
        db = davies_bouldin_score(X_space[mask_valid], labels_pred[mask_valid])
        ch = calinski_harabasz_score(X_space[mask_valid], labels_pred[mask_valid])
    else:
        sil, db, ch = np.nan, np.nan, np.nan

    return {
        "Model": model_name,
        "Space": space_name,
        "N clusters found": n_clusters,
        "N noise points": n_noise,
        "Silhouette Score": sil,
        "Davies-Bouldin Index": db,
        "Calinski-Harabasz": ch,
        "ARI": adjusted_rand_score(y_true, labels_pred) if n_clusters >= 1 else np.nan,
        "NMI": normalized_mutual_info_score(y_true, labels_pred) if n_clusters >= 1 else np.nan,
        "Homogeneity": homogeneity_score(y_true, labels_pred) if n_clusters >= 1 else np.nan,
        "Completeness": completeness_score(y_true, labels_pred) if n_clusters >= 1 else np.nan,
    }


clustering_rows = []

silhouette_by_k = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    labels = km.fit_predict(X_pca95)
    sil = silhouette_score(X_pca95, labels)
    silhouette_by_k.append((k, km.inertia_, sil))

df_k = pd.DataFrame(silhouette_by_k, columns=["k", "inertia", "silhouette"])
display(df_k.round(4))

best_k_sil = int(df_k.sort_values("silhouette", ascending=False).iloc[0]["k"])

for k, name in [(4, "K-Means (K=4)"), (best_k_sil, f"K-Means (K={best_k_sil}, best Sil)")]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    labels = km.fit_predict(X_pca95)
    clustering_rows.append(evaluate_clustering(X_pca95, y_encoded, labels, name, "PCA 95% features"))

eps_values = [5.0, 10.0, 20.0, 30.0, 50.0]
min_samples_values = [5, 20, 50, 100]
dbscan_candidates = []

for eps in eps_values:
    for ms in min_samples_values:
        dbs = DBSCAN(eps=eps, min_samples=ms)
        labels = dbs.fit_predict(X_pca95)
        row = evaluate_clustering(X_pca95, y_encoded, labels, f"DBSCAN (eps={eps}, ms={ms})", "PCA 95% features")
        row["eps"] = eps
        row["min_samples"] = ms
        dbscan_candidates.append(row)

dbscan_df = pd.DataFrame(dbscan_candidates)
display(dbscan_df.sort_values("ARI", ascending=False, na_position="last").head(10).round(4))

valid_dbscan = dbscan_df[dbscan_df["N clusters found"] >= 2].copy()
if len(valid_dbscan) > 0:
    best_dbscan = valid_dbscan.sort_values("ARI", ascending=False).iloc[0].drop(labels=["eps", "min_samples"]).to_dict()
    clustering_rows.append(best_dbscan)

for linkage_method in ["ward", "complete", "average"]:
    agg = AgglomerativeClustering(n_clusters=4, linkage=linkage_method)
    labels = agg.fit_predict(X_pca95)
    clustering_rows.append(evaluate_clustering(
        X_pca95,
        y_encoded,
        labels,
        f"Agglomerative (K=4, {linkage_method})",
        "PCA 95% features",
    ))

clustering_results_df = pd.DataFrame(clustering_rows)
clustering_results_df = clustering_results_df.sort_values("ARI", ascending=False, na_position="last").reset_index(drop=True)
display(clustering_results_df.round(4))

clustering_output = OUTPUT_DIR / "clustering_comparison_generated.csv"
clustering_results_df.to_csv(clustering_output, index=False)
print("Resultados de clustering generados en:", clustering_output)

plt.figure(figsize=(10, 5))
sns.barplot(data=clustering_results_df, x="ARI", y="Model")
plt.title("Comparación de clustering por ARI")
plt.tight_layout()
plt.show()


**Análisis no supervisado.** Si K-Means o Agglomerative alcanzan ARI moderado, existe estructura latente parcialmente consistente con las clases reales. Si DBSCAN produce mucho ruido o bajo ARI, esto indica que los grupos no se comportan como regiones de densidad claramente separadas.


## 10. Comparación global

Se integran los modelos supervisados clásicos, la DNN densa y un clasificador con PCA como preprocesamiento para comparar desempeño.


In [ ]:
supervised_summary = (
    model_results_df[model_results_df["feature_set"] == "all_features"]
    .rename(columns={
        "model": "Model",
        "accuracy": "Accuracy",
        "precision_macro": "Precision macro",
        "recall_macro": "Recall macro",
        "specificity_macro": "Specificity macro",
        "f1_macro": "F1 macro",
        "n_features": "Number of input variables",
    })
)
supervised_summary["Input type"] = "Handcrafted features"

global_rows = supervised_summary[[
    "Model", "Input type", "Number of input variables", "Accuracy",
    "Precision macro", "Recall macro", "Specificity macro", "F1 macro",
]].copy()

if RUN_DNN and len(dnn_metrics) > 0:
    global_rows = pd.concat([global_rows, dnn_metrics], ignore_index=True)

RUN_PCA_CLASSIFIER = True

if RUN_PCA_CLASSIFIER:
    pca_classifier = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_95, random_state=SEED)),
        ("model", SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", random_state=SEED)),
    ])
    pca_classifier.fit(X_train, y_train)
    y_pred_pca = pca_classifier.predict(X_test)
    pca_row = pd.DataFrame([{
        "Model": f"SVM RBF + PCA ({n_95} PC)",
        "Input type": f"PCA ({n_95} components)",
        "Number of input variables": n_95,
        "Accuracy": accuracy_score(y_test, y_pred_pca),
        "Precision macro": precision_score(y_test, y_pred_pca, average="macro", zero_division=0),
        "Recall macro": recall_score(y_test, y_pred_pca, average="macro", zero_division=0),
        "Specificity macro": specificity_macro_score(y_test, y_pred_pca, labels=np.arange(len(class_names))),
        "F1 macro": f1_score(y_test, y_pred_pca, average="macro", zero_division=0),
    }])
    global_rows = pd.concat([global_rows, pca_row], ignore_index=True)

global_comparison = global_rows.sort_values("F1 macro", ascending=False).reset_index(drop=True)
display(global_comparison.round(5))

global_output = OUTPUT_DIR / "final_comparison_all_models_generated.csv"
global_comparison.to_csv(global_output, index=False)
print("Comparación global generada en:", global_output)

plt.figure(figsize=(10, 5))
sns.barplot(data=global_comparison.head(10), x="F1 macro", y="Model", hue="Input type", dodge=False)
plt.xlim(0, 1.05)
plt.title("Comparación global por F1 macro")
plt.tight_layout()
plt.show()


## 11. Conclusiones

- El análisis estadístico permite verificar si las imágenes contienen diferencias cuantificables entre clases.
- Las features handcrafted combinan información de color, textura, morfología, bordes, frecuencia e intensidad.
- Los modelos clásicos, especialmente SVM y Random Forest, suelen ser competitivos cuando las variables están bien diseñadas.
- La DNN densa evalúa un enfoque neuronal sin usar CNN ni imágenes crudas.
- PCA permite reducir dimensionalidad y explorar estructura latente.
- Los métodos no supervisados son útiles para exploración, pero no reemplazan a los modelos supervisados para clasificación final.
- En una aplicación real, el sistema debe considerarse apoyo diagnóstico o segunda opinión, no sustituto del patólogo ni de la citometría de flujo.


## 12. Limitaciones y trabajo futuro

- Dataset limitado a una fuente específica.
- Posible dependencia de condiciones de adquisición, iluminación, cámara, tinción y microscopio.
- Riesgo de sobreajuste si no se valida externamente.
- Necesidad de probar con imágenes de otros hospitales y laboratorios.
- Extensión futura natural: CNN, transfer learning, validación clínica y explicabilidad visual.


## 13. Referencias

- Kaggle: [mehradaria/leukemia](https://www.kaggle.com/datasets/mehradaria/leukemia/data?select=Segmented)
- Repositorio asociado: [MehradAria/ALL-Subtype-Classification](https://github.com/MehradAria/ALL-Subtype-Classification)
- Paper: *A Fast and Efficient CNN Model for B-ALL Diagnosis and its Subtypes Classification using Peripheral Blood Smear Images*
